<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=343559187" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD, NEW SESSION: environment through Stage 11, APTOS, RESNET50 =====
# This is the LAST missing architecture for APTOS. Custom CNN, EfficientNetB0, and
# MobileNetV2 are already done on this source. Once this completes, APTOS is fully closed.

!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# ---------- CONFIG ----------
APTOS_CSV     = '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv'
APTOS_IMG     = '/kaggle/input/competitions/aptos2019-blindness-detection/train_images'

GRADES      = ['0','1','2','3','4']
IMG_SIZE, BATCH_SIZE = 224, 32
PHASE1_EPOCHS, PHASE1_LR, PHASE2_LR, EARLYSTOP_PAT, MONITOR = 10, 1e-3, 1e-5, 7, 'val_accuracy'
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)

# ---------- DATA REBUILD, APTOS ONLY ----------
aptos = pd.read_csv(APTOS_CSV)
aptos['grade']      = aptos['diagnosis'].astype(int).astype(str)
aptos['image_path'] = APTOS_IMG + '/' + aptos['id_code'].astype(str) + '.png'
aptos['source']     = 'aptos'
aptos['patient_id'] = None

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, falling back to unstratified.")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_image_level(df, rs=SEED, tag=""):
    tr, tmp = safe_split(df, 'grade', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

a_tr, a_va, a_te = split_image_level(aptos, tag="APTOS")

cls = np.array(GRADES)
cw = compute_class_weight('balanced', classes=cls, y=a_tr['grade'])
aptos_class_weight = {i: w for i, w in enumerate(cw)}
span = cw.max()/cw.min()
print(f"\nAPTOS class weight span: {span:.1f}x (expect ~9.4x)")

# ================================================================
# STAGE 11: APTOS, RESNET50 (last architecture for this source)
# ================================================================

def make_source_gens(preprocess_fn, tr_df, va_df, te_df):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='grade', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=GRADES, color_mode='rgb')
    tr = train_idg.flow_from_dataframe(tr_df, shuffle=True,  seed=SEED, **common)
    va = eval_idg.flow_from_dataframe(va_df,  shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(te_df,  shuffle=False, **common)
    return tr, va, te

def build_pretrained(base_class, num_classes=5, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(),
                         Dense(256,activation='relu'), Dropout(0.3),
                         Dense(num_classes,activation='softmax')])
    return model, base

tag_p1 = "ss_res_aptos_dr_phase1"
tag_p2 = "ss_res_aptos_dr"
tr, va, te = make_source_gens(res_pre, a_tr, a_va, a_te)
model, base = build_pretrained(ResNet50)

base.trainable = False
model.compile(Adam(PHASE1_LR), 'categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
print(f"\n===== res, aptos: PHASE 1 (head only, {PHASE1_EPOCHS} epochs) =====")
model.fit(tr, validation_data=va, epochs=PHASE1_EPOCHS, class_weight=aptos_class_weight,
          callbacks=[ModelCheckpoint(f'/kaggle/working/{tag_p1}.keras', monitor=MONITOR, save_best_only=True),
                     CSVLogger(f'/kaggle/working/{tag_p1}_log.csv', append=False)], verbose=1)
print("res, aptos Phase 1 saved.")

base.trainable = True
model.compile(Adam(PHASE2_LR), 'categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc', multi_label=False)])
print(f"\n===== res, aptos: PHASE 2 (full fine-tune, up to 60 epochs) =====")
model.fit(tr, validation_data=va, epochs=60, class_weight=aptos_class_weight,
          callbacks=[EarlyStopping(monitor=MONITOR, patience=EARLYSTOP_PAT, restore_best_weights=True),
                     ModelCheckpoint(f'/kaggle/working/{tag_p2}.keras', monitor=MONITOR, save_best_only=True),
                     CSVLogger(f'/kaggle/working/{tag_p2}_log.csv', append=False)], verbose=1)
result = model.evaluate(te, verbose=0)
print(f"\n{tag_p2} TEST: loss={result[0]:.4f} accuracy={result[1]:.4f} auc={result[2]:.4f}")
print("res, aptos Phase 2 saved.")

pd.DataFrame([{'arch':'res','source':'aptos','loss':result[0],'accuracy':result[1],'auc':result[2]}]) \
  .to_csv('/kaggle/working/dr_stage11_aptos_res.csv', index=False)
print(f"\nSaved: {tag_p1}.keras, {tag_p2}.keras, dr_stage11_aptos_res.csv")
print("\n===== APTOS SOURCE COMPLETE: all four architectures done =====")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 118.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.19.0 which is incompatible.
tf-keras 2.20.0 requires tensorflow<2.21,>=2.20, but you have tensorflow 2.19.0 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.19.0 which is incompatible.


2026-08-19 22:08:16.635119: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787177296.659083      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787177296.666652      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787177296.685689      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787177296.685710      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787177296.685712      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras

APTOS class weight span: 9.4x (expect ~9.4x)
Found 2563 validated image filenames belonging to 5 classes.
Found 549 validated image filenames belonging to 5 classes.
Found 550 validated image filenames belonging to 5 classes.


I0000 00:00:1787177315.795541      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787177315.801503      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


94765736/94765736 [==============================] - 1s 0us/step

===== res, aptos: PHASE 1 (head only, 10 epochs) =====
Epoch 1/10


I0000 00:00:1787177331.756530      73 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1787177334.565350      75 service.cc:152] XLA service 0x79f484ea98d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787177334.565393      75 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787177334.565397      75 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787177334.727516      75 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


81/81 [==============================] - 439s 5s/step - loss: 1.3942 - accuracy: 0.5728 - auc: 0.8433 - val_loss: 0.9830 - val_accuracy: 0.5865 - val_auc: 0.8730
Epoch 2/10
81/81 [==============================] - 333s 4s/step - loss: 1.0625 - accuracy: 0.6449 - auc: 0.9073 - val_loss: 0.7869 - val_accuracy: 0.6940 - val_auc: 0.9189
Epoch 3/10
81/81 [==============================] - 334s 4s/step - loss: 0.9964 - accuracy: 0.6812 - auc: 0.9206 - val_loss: 0.7920 - val_accuracy: 0.6448 - val_auc: 0.9199
Epoch 4/10
81/81 [==============================] - 334s 4s/step - loss: 0.9374 - accuracy: 0.7035 - auc: 0.9319 - val_loss: 0.8512 - val_accuracy: 0.6375 - val_auc: 0.9079
Epoch 5/10
81/81 [==============================] - 336s 4s/step - loss: 0.9312 - accuracy: 0.7046 - auc: 0.9333 - val_loss: 0.7199 - val_accuracy: 0.7031 - val_auc: 0.9350
Epoch 6/10
81/81 [==============================] - 334s 4s/step - loss: 0.9152 - accuracy: 0.7148 - auc: 0.9375 - val_loss: 0.7107 - val_accuracy